# Pre-Label Real Plans on GPU (Step 1 of fine-tuning)

Runs the current door detector over real drawing sets and writes an image + YOLO-format label file per floor-plan page. You then **correct** those boxes in Roboflow instead of drawing hundreds from scratch (3-5× faster) — see `docs/FINE_TUNING.md` §3.

Doing it here instead of locally because detection over hundreds of pages is far faster on a T4, **and** the final cell can push images + annotations straight to Roboflow via its API — no download-then-re-upload round trip.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Clone + install

In [ ]:
!git clone https://github.com/pnkkumar123/ocr-project.git
%cd ocr-project/backend
!pip install -q pymupdf pydantic pydantic-settings pillow ultralytics roboflow

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY — set Runtime > T4 GPU')

## 2. Upload the current weights
`backend/models/*.pt` is gitignored, so upload `door_yolo11.pt` from your machine.

In [ ]:
import os
from google.colab import files

os.makedirs('models', exist_ok=True)
print('Upload door_yolo11.pt (from backend/models/):')
for name in files.upload():
    os.replace(name, f'models/{name}')
print(os.listdir('models'))

## 3. Upload the drawing sets

Upload as many *different* sources as you can — style diversity is what stops the model overfitting to one firm. Aim to mix: commercial + residential, different architects, and at least one **scanned/raster** set (that's what teaches it to survive a photocopied PDF).

Large files take a while; Drive-mount (next cell) is nicer if you'll re-run.

In [ ]:
os.makedirs('../datasets/pdfs', exist_ok=True)
print('Upload one or more drawing-set PDFs:')
for name in files.upload():
    os.replace(name, f'../datasets/pdfs/{name}')
print(os.listdir('../datasets/pdfs'))

In [ ]:
# Alternative: keep PDFs in Drive so you don't re-upload every session.
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/fire-door-pdfs/*.pdf ../datasets/pdfs/

## 4. Pre-label on GPU

Two-stage: a cheap drawing-density scan picks `--candidates` likely floor-plan pages per PDF (no rendering, no model), then detection runs only on those and keeps the top `--max-pages` by door count. That's what makes an 800-page set tractable.

`--conf 0.15` is deliberately lower than production's 0.25: when pre-labeling you want **recall**, because deleting a wrong box is one click but spotting a door the model never proposed is slow.

In [ ]:
!python scripts/prelabel.py ../datasets/pdfs/*.pdf \
    --device cuda --conf 0.15 --max-pages 5 --candidates 14

In [ ]:
# What we produced
from pathlib import Path
inbox = Path('../datasets/label_inbox')
pngs = sorted(inbox.glob('*.png'))
boxes = sum(len(p.with_suffix('.txt').read_text().strip().splitlines())
            for p in pngs if p.with_suffix('.txt').exists())
print(f'{len(pngs)} pages, {boxes} pre-labeled boxes')
for p in pngs:
    n = len(p.with_suffix('.txt').read_text().strip().splitlines())
    print(f'  {p.name}: {n}')

## 5. Visual sanity check

**Do this before investing hours in labeling.** Confirm the selected pages are genuinely floor plans (not schedules or detail sheets) and that boxes land on door swing arcs. Expect some wrong boxes — deleting them *is* the labeling job.

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

for p in pngs[:4]:
    img = Image.open(p).convert('RGB')
    W, H = img.size
    d = ImageDraw.Draw(img)
    for line in p.with_suffix('.txt').read_text().strip().splitlines():
        _, cx, cy, w, h = line.split()
        cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
        d.rectangle([cx-w/2, cy-h/2, cx+w/2, cy+h/2], outline=(255, 0, 0), width=6)
    print(p.name)
    display(img.resize((1100, int(1100 * H / W))))

## 6a. Push straight to Roboflow (recommended)

Uploads images **with** the pre-labels attached, so you open Roboflow and start correcting immediately — no zip, no download, no re-upload.

First create the project: roboflow.com → **Create New Project** → **Object Detection**, annotation group `door`. Then fill in the two values below (API key: Settings → API → Private API Key).

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = ''      # Settings -> API -> Private API Key
WORKSPACE        = ''      # e.g. 'my-workspace' (from your project URL)
PROJECT          = 'fire-door-real-plans'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)

for p in pngs:
    label = p.with_suffix('.txt')
    project.upload(
        image_path=str(p),
        annotation_path=str(label) if label.exists() else None,
        split='train',
    )
    print('uploaded', p.name)
print('\nDone — open the project in Roboflow and start correcting boxes.')

## 6b. Or download a zip instead
Use this if you'd rather upload to Roboflow by hand (or use a different labeling tool).

In [ ]:
import shutil
shutil.make_archive('/content/label_inbox', 'zip', inbox)
files.download('/content/label_inbox.zip')

## Next
1. Correct the boxes in Roboflow (`docs/FINE_TUNING.md` §3 for conventions — box the swing arc + leaf together; skip legend/detail doors; be consistent about double doors).
2. **Versions → Create New Version** (70/20/10 split, light augmentation only — no colour jitter on line art).
3. Run `colab/finetune_door_detector.ipynb`.